# Plan 005.2 — Cross-catalog content comparison

This walkthrough uses the implemented public API to compare **known current content**
in two temporary catalogs. It is for readers familiar with scanning a directory and
Python context managers. Run with the repository's `uv` environment; no personal files
or external services are used. Run cells in order and finish with the cleanup cell.

You will build overlapping inventories, inspect complete summaries and bounded
identity batches, reuse captured snapshots, and see why cataloged presence differs
from physical verification. The final examples cover targeted removal and reopen.

References: [Plan 005.2](../../docs/plans/005.2-cross-catalog-content-comparison.md),
[Catalog API](../../docs/catalog-api.md#compare_content), and
[targeted updates](plan-005.1-targeted-catalog-updates.ipynb).

## 1. Build a disposable example

The source has four paths but three identities: shared bytes, missing bytes at two
paths, and an empty file. The target has the shared bytes under a different name
plus target-only bytes. Database files live outside the scanned roots.

In [ ]:
from contextlib import ExitStack, closing
from dataclasses import asdict
from pathlib import Path, PurePosixPath
from tempfile import TemporaryDirectory

from archiver import Catalog, ContentComparisonError, ContentComparisonSummary

resources = ExitStack()
base = Path(resources.enter_context(TemporaryDirectory(prefix="archiver-0052-")))
source_root, target_root = base / "source", base / "target"
source_root.mkdir()
target_root.mkdir()
(source_root / "a").mkdir()
(source_root / "a/nested.bin").write_bytes(b"missing")
(source_root / "a.zip").write_bytes(b"missing")
(source_root / "empty.bin").write_bytes(b"")
(source_root / "shared.txt").write_bytes(b"shared")
(target_root / "renamed.txt").write_bytes(b"shared")
(target_root / "target-only.txt").write_bytes(b"target-only")

source = resources.enter_context(Catalog.create(base / "source.sqlite"))
target = resources.enter_context(Catalog.create(base / "target.sqlite"))
source.scan_directory(source_root)
target.scan_directory(target_root)
assert source.files(source_root).count() == 4
assert target.files(target_root).count() == 2
print("Source: 4 paths; target: 2 paths. Both catalogs use schema", source.schema_version)

## 2. Inspect complete summary counts and captured scope

Counts describe distinct identities, so duplicates count once. The two missing
identities require seven unique bytes: seven bytes for `missing` and zero for the
empty file. The shared identity contributes six bytes to neither missing total.

In [ ]:
def physical_state(root: Path) -> dict:
    return {
        path.relative_to(root).as_posix():
        (path.read_bytes(), path.stat().st_mtime_ns, path.stat().st_mode)
        for path in root.rglob("*") if path.is_file()
    }

files_before = (physical_state(source_root), physical_state(target_root))
databases_before = (source.database_path.read_bytes(), target.database_path.read_bytes())

with source.compare_content(target, source_root=source_root, target_root=target_root) as comparison:
    summary = comparison.summary()
    source_scope, target_scope = comparison.source_scope, comparison.target_scope
    assert summary == ContentComparisonSummary(3, 2, 1, 2, 1, 7)
    assert source_scope.has_inventory and target_scope.has_inventory
    assert source_scope.catalog_uuid == source.catalog_uuid
    assert source_scope.root == source_root.resolve()
    print(asdict(summary))
    print("Both scopes have successful inventories:", source_scope.has_inventory, target_scope.has_inventory)

`revision` is an opaque baseline token. Location IDs are local to their catalog;
use the scope's UUID, root, and revision rather than comparing database row IDs.
Returned records remain available after the session closes, while SQL-backed
methods require an open session.

## 3. Read missing content in bounded batches

The default emits missing source identities, ordered by full algorithm/digest.
The representative path is the smallest full POSIX path using SQLite BINARY
ordering: `a.zip` wins over `a/nested.bin`. It is a candidate path, not a newly
verified file. Production consumers should handle each batch without retaining
earlier batches; the list below is only a small teaching fixture.

In [ ]:
with source.compare_content(target, source_root=source_root, target_root=target_root) as comparison:
    missing_rows = []
    for batch in comparison.batches(batch_size=1):
        assert isinstance(batch, tuple) and len(batch) <= 1
        missing_rows.extend(batch)
        for row in batch:
            print(row.source_relative_path, "bytes:", row.size_bytes,
                  "source copies:", row.source_instance_count, "in target:", row.in_target)
    assert len(missing_rows) == 2
    duplicate = next(row for row in missing_rows if row.size_bytes == 7)
    assert duplicate.source_relative_path == PurePosixPath("a.zip")
    assert duplicate.source_instance_count == 2
    assert duplicate.source_mtime_ns == (source_root / "a.zip").stat().st_mtime_ns
    assert all(not row.in_target for row in missing_rows)
    assert comparison.summary() == summary  # batch bounds do not limit summary counts

## 4. Classify all source content and replay safely

`include_present=True` includes shared content with `in_target=True`. A session
supports sequential traversals over its same captured inventories, with one active
iterator at a time. After an early `break`, close the iterator explicitly or leave
the session. Here `closing()` releases its cursor even though we stop early.

In [ ]:
with source.compare_content(target, source_root=source_root, target_root=target_root) as comparison:
    with closing(comparison.batches(batch_size=1, include_present=True)) as batches:
        for batch in batches:
            print("Early preview:", batch[0].source_relative_path)
            break
    classified = tuple(row for batch in comparison.batches(include_present=True) for row in batch)
    replay = tuple(row for batch in comparison.batches(include_present=True) for row in batch)
    assert classified == replay
    assert len(classified) == 3 and sum(row.in_target for row in classified) == 1
    assert [row.content_id.digest for row in classified] == sorted(row.content_id.digest for row in classified)
    print([(str(row.source_relative_path), row.in_target) for row in classified])

try:
    comparison.summary()
except ContentComparisonError:
    print("Closed-session reads fail clearly; captured summary values still work.")
else:
    raise AssertionError("A closed comparison must reject SQL-backed reads")
assert summary.missing_source_content_count == 2

## 5. Same-root comparison and a direction exercise

Comparing a catalog/root with itself produces no missing content. Reversing the
source and target changes proposed work.

**Exercise:** predict the reverse comparison before running the next cell.
How many source identities will be missing, and how many unique bytes is that?
The target-only file contains eleven bytes.

In [ ]:
with source.compare_content(source, source_root=source_root, target_root=source_root) as same:
    assert same.summary() == ContentComparisonSummary(3, 3, 3, 0, 0, 0)
    assert list(same.batches()) == []

# Answer: the reverse source has two identities; only target-only.txt is missing.
with target.compare_content(source, source_root=target_root, target_root=source_root) as reverse:
    assert reverse.summary() == ContentComparisonSummary(2, 3, 1, 1, 2, 11)
    print(asdict(reverse.summary()))

assert (physical_state(source_root), physical_state(target_root)) == files_before
assert (source.database_path.read_bytes(), target.database_path.read_bytes()) == databases_before
print("Comparison preserved physical file bytes/mtime/mode and both database files.")

## 6. Cataloged presence is not physical verification

We now deliberately remove **one synthetic target file** to simulate a stale
inventory. This setup action is outside comparison. A new comparison still reports
the cataloged shared identity: comparison does not scan or hash physical files.

In [ ]:
(target_root / "renamed.txt").unlink()  # synthetic fixture change, not a Catalog operation
with source.compare_content(target, source_root=source_root, target_root=target_root) as stale:
    assert stale.summary() == summary
    print("Catalog still reports shared identities:", stale.summary().shared_content_count)

## 7. Publish a targeted inventory removal and reopen

Close comparison readers before publishing or pruning. We explicitly retire the
missing target path with the completed 005.1 API, then open a new comparison.
The historical identity remains known in `contents()`, but it no longer contributes
to current target presence. No full scan is needed.

In [ ]:
changes = target.prepare_file_update(target_root, remove_paths=[PurePosixPath("renamed.txt")])
update = target.apply_inventory_changes(changes)
assert update.removed_files == 1 and update.origin == "targeted_update"
assert target.contents().count() == 2  # shared content remains catalog metadata
assert target.files(target_root).count() == 1

with Catalog.open(target.database_path) as reopened:
    with source.compare_content(reopened, source_root=source_root, target_root=target_root) as current:
        assert current.summary() == ContentComparisonSummary(3, 1, 0, 3, 1, 13)
        assert current.target_scope.revision != target_scope.revision
        print(asdict(current.summary()))
assert physical_state(source_root) == files_before[0]

## 8. An unscanned root is an empty known set

`has_inventory=False` means no successful inventory was captured. It does not mean
the physical directory is empty. Comparing this root creates no location or scan.

In [ ]:
unknown_root = base / "unscanned"
unknown_root.mkdir()
(unknown_root / "unobserved.txt").write_bytes(b"shared")
with source.compare_content(target, source_root=source_root, target_root=unknown_root) as unknown:
    assert not unknown.target_scope.has_inventory
    assert unknown.target_scope.location.id is None
    assert unknown.summary() == ContentComparisonSummary(3, 0, 0, 3, 0, 13)
    print("No inventory captured; all three source identities are missing from the known target set.")

## Implementation boundaries and cleanup

The implementation uses a dedicated read-only SQLite connection, grouped identity
relations, SQL summaries and bounded fetches. It does not build whole-catalog Python
sets. SQLite may still need inventory-sized grouping/sorting storage. Requested
batch sizes are upper bounds; individual batches are capped at 10,000 rows.

A read transaction preserves captured inventories for sequential traversals. Two
catalogs form a stable, independently captured pair, not a globally atomic instant.
WAL can permit concurrent writers; rollback journaling can block them until readers
close. Close comparisons before mutation or retention. Opening identity checks
cannot protect against every concurrent database-file replacement.

Common pitfall: treating `in_target` as proof that bytes still exist. Use the
cataloged-presence meaning shown above; later synchronization must separately
verify destination bytes. Workspace construction, copying and recovery remain
later-plan work. Optional exercise: create another source path with duplicate
bytes, publish a targeted update, and check that instance counts increase while
distinct counts stay unchanged.

Run the last cell to close both catalogs and remove the entire temporary sandbox.

In [ ]:
resources.close()
assert not base.exists()
print("Catalogs closed and temporary files removed.")